In [0]:
from src.utils.load_config import paths_config as pc
from src.utils.load_config import sample_data_path
from os import listdir

catalog_name = pc['catalog']

raw_schema = pc['schemas']['raw']
refined_schema = pc['schemas']['refined']
analytics_schema = pc['schemas']['analytics']

landing_volume = pc['volumes']['landing']
checkpoint_volume = pc['volumes']['checkpoint']



In [0]:
def create_catalog(catalog_name: str) -> None:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

def create_schema(catalog_name: str, schema_name: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

def create_volume(catalog_name: str, schema_name: str, volume_name: str) -> None:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")


def create_subpath(catalog_name: str, schema_name: str, volume_name: str, subpath: str) -> None: 
    volume_str = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{subpath}"
    dbutils.fs.mkdirs(volume_str)


In [0]:
create_catalog(pc['catalog'])
[create_schema(pc['catalog'], schema) for schema in pc['schemas'].values()]

for volume in pc['volumes']:
    create_volume(pc['catalog'], pc['schemas']['raw'], volume)
    for subpath in pc['landing'].values():
        create_subpath(pc['catalog'], pc['schemas']['raw'], volume, subpath)



In [0]:

for volume in pc['volumes']:
    for subpath in pc['landing']:
        volume_path = f"/Volumes/{catalog_name}/{raw_schema}/{volume}/{subpath}/"
        origin_path = sample_data_path / subpath
        origin = set(listdir(origin_path))
        dest = set(listdir(volume_path))
        missing_files = origin.difference(dest)
        for missing_file in missing_files:
            file = (origin_path / missing_file).as_posix()
            dbutils.fs.cp(file, volume_path)

        